# 02 - Preprocessing & Feature Engineering

Questa fase trasforma i dati grezzi in feature pronte per il Machine Learning.

**Step:**
1. Caricamento dati puliti da 01_Data_Ingestion_EDA
2. Data Quality Check (duplicati, missing values)
3. Feature Extraction (Coupon, DTM, YTM, Stress interbancario (Euribor 3m - Tasso BCE), Pendenza a breve (Euribor 1Y - 3M), Differenziale BCE-FED)
4. Gestione delle frequenze miste (Forward-Fill, Shift per Lookahead Bias)
5. Creazione Lagged Features per Time-Series
6. Merge e allineamento temporale finale

## 2.1 - Import & Load Cleaned Data

In [1]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import seaborn as sns

# Style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Carica i dataset puliti dal notebook precedente
df_anagrafica_clean = pd.read_csv('./data/df_anagrafica_clean.csv')
df_storico_clean = pd.read_csv('./data/df_storico_clean.csv')
df_macro = pd.read_csv('./data/df_macro.csv', index_col=0, parse_dates=True)

# Assicurati che le date siano datetime
df_storico_clean['referencedate'] = pd.to_datetime(df_storico_clean['referencedate'])
df_anagrafica_clean['redemptiondate'] = pd.to_datetime(df_anagrafica_clean['redemptiondate'], format='%d/%m/%Y', errors='coerce')

print(f"✓ Dataset caricati")
print(f"  Anagrafica: {df_anagrafica_clean.shape}")
print(f"  Storico: {df_storico_clean.shape}")
print(f"  Macro: {df_macro.shape}")

✓ Dataset caricati
  Anagrafica: (305, 10)
  Storico: (189510, 9)
  Macro: (9485, 12)


## 2.2 - Data Quality Check

In [2]:
# 1. Duplicati su ISIN+Data
if 'isincode' in df_storico_clean.columns and 'referencedate' in df_storico_clean.columns:
    n_dupes = df_storico_clean.duplicated(subset=['isincode', 'referencedate']).sum()
    print(f"Duplicati su ISIN+Data nello storico: {n_dupes}")
    
# 2. Missing nelle colonne chiave
print("\nMissing values nelle colonne chiave:")
print(df_storico_clean[['isincode', 'referencedate', 'pricevalue']].isnull().sum())
print("\nAnag missing:")
print(df_anagrafica_clean[['isincode', 'redemptiondate', 'description']].isnull().sum())

Duplicati su ISIN+Data nello storico: 0

Missing values nelle colonne chiave:
isincode         0
referencedate    0
pricevalue       0
dtype: int64

Anag missing:
isincode          0
redemptiondate    0
description       0
dtype: int64


## 2.3 - Feature Extraction: Coupon (Text Mining)

In [3]:
def extract_coupon(description):
    """
    Estrae il tasso di cedola (coupon) dalla descrizione del titolo.
    
    - Zero Coupon: Ritorna 0.0 se contiene 'bot', 'zc', 'zero'
    - Pattern numerico: Cerca "5%" o "3,5%" nella descrizione
    - Pattern EUR: Cerca "EUR 5" nella descrizione
    
    Returns: float (%) oppure np.nan
    """
    if pd.isnull(description):
        return np.nan
    desc = str(description)

    # ZERO COUPON
    if 'bot' in desc.lower() or 'zc' in desc.lower() or 'zero' in desc.lower() or 'ctz' in desc.lower():
        return 0.0

    # Pattern: numeri + %
    match = re.search(r'(\d+[\.,]\d+|\d+)[ ]*%', desc)
    if match:
        return float(match.group(1).replace(',', '.'))
    
    # Pattern: EUR + numeri
    match = re.search(r'[Ee][Uu][Rr][ ]*(\d+[\.,]\d+|\d+)', desc)
    if match:
        return float(match.group(1).replace(',', '.'))
   
    return np.nan

df_anagrafica_clean['coupon'] = df_anagrafica_clean['description'].apply(extract_coupon)

print("--- COUPON EXTRACTION RESULTS ---")
print(f"Coupon estratti: {df_anagrafica_clean['coupon'].notna().sum()}")
print(f"Coupon mancanti: {df_anagrafica_clean['coupon'].isna().sum()}")
print(f"\nCoupon Statistics:")
print(df_anagrafica_clean['coupon'].describe())

# Visualizza alcuni esempi
print("\nEsempi di estrazione:")
display(df_anagrafica_clean[['description', 'coupon']].head(10))

--- COUPON EXTRACTION RESULTS ---
Coupon estratti: 305
Coupon mancanti: 0

Coupon Statistics:
count    305.000000
mean       2.327951
std        1.663578
min        0.000000
25%        0.850000
50%        2.500000
75%        3.450000
max        7.250000
Name: coupon, dtype: float64

Esempi di estrazione:


,description,coupon
0,Btp Fx 3.15% Jun31 Eur,3.15
1,Schatz Fx 2.5% Jun28 Eur,2.50
2,Btp Fx 3.8% Jul36 Eur,3.80
3,Btp Fx 3.3% Jun33 Eur,3.30
4,Bot Zc Apr27 A Eur,0.00
5,Bot Zc Jul26 Q Eur,0.00
6,Bot Zc Sep26 S Eur,0.00
7,Bot Zc Mar27 A Eur,0.00
8,Obligaciones Fx 3.95% Oct56 Eur,3.95
9,Bonos Fx 2.6% May31 Eur,2.60


In [4]:
# Scarta i bond con coupon non estraibile (probabilmente "esotici" sfuggiti al filtro precedente)
n_before = df_anagrafica_clean.shape[0]
df_anagrafica_clean = df_anagrafica_clean.dropna(subset=['coupon']).copy()
print(f"Obbligazioni rimosse per coupon mancante: {n_before - df_anagrafica_clean.shape[0]}")
print(f"Obbligazioni rimanenti: {df_anagrafica_clean.shape[0]}")

Obbligazioni rimosse per coupon mancante: 0
Obbligazioni rimanenti: 305


## 2.4 - Feature Extraction: Days to Maturity (DTM), Yield to Maturity (YTM)

In [5]:
# Merge: Uniamo le feature fisse (cedola, scadenza) al file dei prezzi giornalieri
df_ml = pd.merge(df_storico_clean, 
                 df_anagrafica_clean[['isincode', 'description', 'redemptiondate', 'coupon']], 
                 on='isincode', 
                 how='inner')

print(f"✓ Merge completato: {df_ml.shape[0]} righe")

# DTM = Giorni fino alla scadenza
df_ml['days_to_maturity'] = (df_ml['redemptiondate'] - df_ml['referencedate']).dt.days
df_ml['years_to_maturity'] = df_ml['days_to_maturity'] / 365.25

# Scarta bond già scaduti (DTM <= 0)
n_before = df_ml.shape[0]
df_ml = df_ml[df_ml['days_to_maturity'] > 0].copy()
print(f"Bond scaduti rimossi: {n_before - df_ml.shape[0]}")

print(f"\n--- DTM STATISTICS ---")
print(df_ml[['days_to_maturity', 'years_to_maturity']].describe())

# Preview
print(f"\nSample:")
display(df_ml[['isincode', 'referencedate', 'pricevalue', 'coupon', 'years_to_maturity']].head(10))

✓ Merge completato: 189510 righe
Bond scaduti rimossi: 0

--- DTM STATISTICS ---
       days_to_maturity  years_to_maturity
count     189510.000000      189510.000000
mean        3951.752789          10.819309
std         3649.812827           9.992643
min            7.000000           0.019165
25%         1396.000000           3.822040
50%         2596.000000           7.107461
75%         5434.000000          14.877481
max        18041.000000          49.393566

Sample:


,isincode,referencedate,pricevalue,coupon,years_to_maturity
0,DE0001030708,2023-01-02,83.82,0.0,7.616701
1,DE0001030708,2023-01-03,84.07,0.0,7.613963
2,DE0001030708,2023-01-04,84.62,0.0,7.611225
3,DE0001030708,2023-01-05,84.37,0.0,7.608487
4,DE0001030708,2023-01-06,85.00,0.0,7.605749
5,DE0001030708,2023-01-09,84.98,0.0,7.597536
6,DE0001030708,2023-01-10,84.59,0.0,7.594798
7,DE0001030708,2023-01-11,85.30,0.0,7.592060
8,DE0001030708,2023-01-12,85.67,0.0,7.589322
9,DE0001030708,2023-01-13,85.78,0.0,7.586585


In [6]:
# Yield to Maturity (YTM) ibrido: bisogna distinguere tra zero-coupon e coupon-bearing
# Per i zero-coupon, YTM = (Face Value / Price)^(1/Years to Maturity) - 1
# Per i coupon-bearing, approssimiamo con YTM ≈ (Coupon + (Face Value - Price) / Years to Maturity) / ((Face Value + Price) / 2) 
import numpy as np

def calculate_ytm(row):
    p = row['pricevalue']
    c = row['coupon'] / 100  # Converti da % a decimale
    t = row['years_to_maturity']
    f = 100  # Quasi sempre il valore nominale è 100 per i bond retail

    # Scartiamo i casi impossibili o vicini a scadenza
    if t < 0.8 or p <= 0:
        return np.nan
    
    # Zero-coupon
    if c == 0:
        ytm = ((f / p) ** (1 / t)) - 1
        return ytm * 100  # Converti in percentuale
    
    # Coupon-bearing (approssimazione)
    else:
        # Numeratore: cedola annuale + Capital Gain/Loss annualizzato
        numerator = c + ((f-p) / t)
        # Denominatore: media del valore nominale e del prezzo
        denominator = (f + p) / 2
        ytm_approximated = numerator / denominator
        return ytm_approximated * 100  # Converti in percentuale

df_ml['Yield_to_Maturity'] = df_ml.apply(calculate_ytm, axis=1)

# # Filtra YTM negativi anomali o troppo alti (< -10% o > 50%)
# n_before = df_ml.shape[0]
# df_ml = df_ml[(df_ml['Yield_to_Maturity'] > -10) & (df_ml['Yield_to_Maturity'] < 50)].copy()
# print(f"Scartate {n_before - df_ml.shape[0]} righe con rendimenti matematicamente anomali.")

print("--- Yield to Maturity Statistics ---")
print(df_ml['Yield_to_Maturity'].describe())
print(f"\nYTM negativo: {(df_ml['Yield_to_Maturity'] < 0).sum()}")
print(f"YTM positivo: {(df_ml['Yield_to_Maturity'] >= 0).sum()}")

--- Yield to Maturity Statistics ---
count    185718.000000
mean          0.749999
std           1.585205
min          -5.091635
25%          -0.216024
50%           0.741379
75%           2.008207
max           6.606896
Name: Yield_to_Maturity, dtype: float64

YTM negativo: 59490
YTM positivo: 126228


## 2.5 - Feature Extraction: Stress interbancario (Euribor 3m - Tasso BCE), Pendenza a breve (Euribor 1Y - 3M), Differenziale BCE-FED 

In [7]:
# 1. Stress Interbancario (Euribor 3M - Tasso BCE)
df_macro['interbank_stress_spread'] = df_macro['Euribor_3M'] - df_macro['ECB_Deposit_Rate']

# 2. Pendenza Curva a Breve (Euribor 1Y - Euribor 3M)
df_macro['short_yield_curve_slope'] = df_macro['Euribor_1Y'] - df_macro['Euribor_3M']

# 3. Differenziale BCE-FED (Impatto sul cambio e sui flussi di capitale)
df_macro['bce_fed_spread'] = df_macro['ECB_Deposit_Rate'] - df_macro['FED_Rate']

## 2.6 - Gestione Frequenze Miste: Forward-Fill & Lookahead Bias Prevention

In [8]:
# PROBLEMA 1: L'euribor è mensile, i tassi sono giornalieri.
# Avremo NaN nei giorni in cui l'inflazione non viene pubblicata.
# SOLUZIONE: Forward-Fill (ffill) - copia il dato del mese scorso su tutti i giorni successivi

df_macro['Euribor_3M'] = df_macro['Euribor_3M'].ffill()
df_macro['Euribor_1Y'] = df_macro['Euribor_1Y'].ffill()

print("✓ Forward-Fill applicato ai dati macro")
print(f"Missing values DOPO ffill: {df_macro.isna().sum().sum()}")

# PROBLEMA 2: Lookahead Bias
# L'inflazione di Marzo viene pubblicata il 15 Aprile.
# Se non shiftiamo, diremmo alla rete neurale il tasso FUTURO.
# SOLUZIONE: Shift di 30 giorni per l'inflazione (delay della data di pubblicazione)

df_macro['HICP_Core_Lagged'] = df_macro['HICP_Core'].shift(30)
df_macro = df_macro.drop(columns=['HICP_Core']).dropna()

df_macro['HICP_Headline_Lagged'] = df_macro['HICP_Headline'].shift(30)
df_macro = df_macro.drop(columns=['HICP_Headline']).dropna()

print("✓ Lookahead Bias Prevention: HICP_Headline shifter di 30 giorni")
print(f"Dataset macro dopo pulizia: {df_macro.shape}")

✓ Forward-Fill applicato ai dati macro
Missing values DOPO ffill: 34588
✓ Lookahead Bias Prevention: HICP_Headline shifter di 30 giorni
Dataset macro dopo pulizia: (4225, 15)


In [9]:
# Preview dati macro dopo preprocessing
display(df_macro.head(40))
display(df_macro.tail())

,ECB_Deposit_Rate,ECB_MRO_Rate,FED_Rate,ESI_Index,Euribor_3M,Euribor_1Y,VIX,STOXX50,XEON,SEGA,interbank_stress_spread,short_yield_curve_slope,bce_fed_spread,HICP_Core_Lagged,HICP_Headline_Lagged
2010-03-01,0.25,1.0,0.14,-12.8,0.644957,3.986665,19.26,2772.699951,138.253006,91.100906,0.394957,3.341709,0.11,94.24,92.0
2010-03-02,0.25,1.0,0.14,-12.8,0.644957,3.986665,19.06,2796.330078,138.257996,91.109749,0.394957,3.341709,0.11,94.24,92.0
2010-03-03,0.25,1.0,0.15,-12.8,0.644957,3.986665,18.83,2822.590088,138.259003,91.153778,0.394957,3.341709,0.10,94.24,92.0
2010-03-04,0.25,1.0,0.16,-12.8,0.644957,3.986665,18.72,2816.100098,138.257004,91.030472,0.394957,3.341709,0.09,94.24,92.0
2010-03-05,0.25,1.0,0.17,-12.8,0.644957,3.986665,17.42,2877.439941,138.257996,91.065689,0.394957,3.341709,0.08,94.24,92.0
2010-03-08,0.25,1.0,0.15,-12.8,0.644957,3.986665,17.79,2879.290039,138.257996,91.083305,0.394957,3.341709,0.10,94.24,92.0
2010-03-09,0.25,1.0,0.14,-12.8,0.644957,3.986665,17.92,2880.709961,138.257996,91.233017,0.394957,3.341709,0.11,94.24,92.0
2010-03-10,0.25,1.0,0.14,-12.8,0.644957,3.986665,18.57,2909.399902,138.259995,91.233017,0.394957,3.341709,0.11,94.24,92.0
2010-03-11,0.25,1.0,0.15,-12.8,0.644957,3.986665,18.06,2895.739990,138.263000,91.206596,0.394957,3.341709,0.10,94.24,92.0
2010-03-12,0.25,1.0,0.17,-12.8,0.644957,3.986665,17.58,2898.360107,138.268997,91.092102,0.394957,3.341709,0.08,94.24,92.0


,ECB_Deposit_Rate,ECB_MRO_Rate,FED_Rate,ESI_Index,Euribor_3M,Euribor_1Y,VIX,STOXX50,XEON,SEGA,interbank_stress_spread,short_yield_curve_slope,bce_fed_spread,HICP_Core_Lagged,HICP_Headline_Lagged
2026-05-04,2.0,2.15,3.64,-12.4,2.027727,3.220984,18.29,5763.609863,149.038895,108.430000,0.027727,1.193257,-1.64,126.51,133.83
2026-05-05,2.0,2.15,3.64,-12.4,2.027727,3.220984,17.38,5869.629883,149.044296,108.669998,0.027727,1.193257,-1.64,126.51,133.83
2026-05-06,2.0,2.15,3.64,-12.4,2.027727,3.220984,17.39,6027.129883,149.044098,109.290001,0.027727,1.193257,-1.64,126.51,133.83
2026-05-07,2.0,2.15,3.63,-12.4,2.027727,3.220984,17.08,5972.649902,149.058899,109.320000,0.027727,1.193257,-1.63,126.51,133.83
2026-05-08,2.0,2.15,3.63,-12.4,2.027727,3.220984,17.19,5972.649902,149.058899,109.320000,0.027727,1.193257,-1.63,126.51,133.83


## 2.7 - Merge Data dei Bond con Macro-economici

In [10]:
# Prepara il DataFrame macro per il merge
df_macro_reset = df_macro.reset_index()
df_macro_reset.columns = df_macro_reset.columns.str.lower()
# Crea la colonna 'referencedate' a partire dall'indice reset (qui chiamato 'index') e rimuovi l'originale
df_macro_reset['referencedate'] = pd.to_datetime(df_macro_reset['index'])
df_macro_reset = df_macro_reset.drop(columns=['index'])

# Assicurati che le date siano datetime
df_ml['referencedate'] = pd.to_datetime(df_ml['referencedate'])

# Merge: left join per mantenere tutte le righe di df_ml e aggiungere i dati macro corrispondenti
df_ml_macro = pd.merge(df_ml, df_macro_reset, 
                        left_on='referencedate', right_on='referencedate', 
                        how='left')

# Scarta le righe con dati macro mancanti (i giorni prima del primo dato macro)
#df_ml_macro = df_ml_macro.dropna(subset=['ECB_Deposit_Rate', 'FED_Rate', 'HICP_Core_Lagged', 'HICP_Headline_Lagged']).copy()

print(f"✓ Dataset finale con feature macroeconomiche: {df_ml_macro.shape}")
print(f"\nColonne disponibili:")
print(df_ml_macro.columns.tolist())

✓ Dataset finale con feature macroeconomiche: (189510, 30)

Colonne disponibili:
['isincode', 'marketcode', 'referencedate', 'endvaluedate', 'pricetype', 'pricevalue', 'volume', 'mintoday', 'maxtoday', 'description', 'redemptiondate', 'coupon', 'days_to_maturity', 'years_to_maturity', 'Yield_to_Maturity', 'ecb_deposit_rate', 'ecb_mro_rate', 'fed_rate', 'esi_index', 'euribor_3m', 'euribor_1y', 'vix', 'stoxx50', 'xeon', 'sega', 'interbank_stress_spread', 'short_yield_curve_slope', 'bce_fed_spread', 'hicp_core_lagged', 'hicp_headline_lagged']


In [11]:
# Preview
print("--- DATASET MACRO-INTEGRATED ---")
display(df_ml_macro.head())
display(df_ml_macro.tail())

print(f"\nStatistiche:")
print(df_ml_macro.describe())

--- DATASET MACRO-INTEGRATED ---


,isincode,marketcode,referencedate,endvaluedate,pricetype,pricevalue,volume,mintoday,maxtoday,description,...,euribor_1y,vix,stoxx50,xeon,sega,interbank_stress_spread,short_yield_curve_slope,bce_fed_spread,hicp_core_lagged,hicp_headline_lagged
0,DE0001030708,MOT,2023-01-02,2023-01-03,RP,83.82,0,0.0,0.0,Bund Green Bond Tf 0% Ag30 Eur,...,2.99956,21.67,3793.620117,135.159897,99.571831,0.063476,0.936084,-2.33,114.56,123.26
1,DE0001030708,MOT,2023-01-03,2023-01-04,RP,84.07,0,0.0,0.0,Bund Green Bond Tf 0% Ag30 Eur,...,2.99956,22.90,3882.290039,135.124496,100.875557,0.063476,0.936084,-2.33,114.56,123.26
2,DE0001030708,MOT,2023-01-04,2023-01-05,RP,84.62,0,0.0,0.0,Bund Green Bond Tf 0% Ag30 Eur,...,2.99956,22.01,3973.969971,135.164993,101.682190,0.063476,0.936084,-2.33,114.56,123.26
3,DE0001030708,MOT,2023-01-05,2023-01-06,RP,84.37,0,0.0,0.0,Bund Green Bond Tf 0% Ag30 Eur,...,2.99956,22.46,3959.479980,135.165802,101.410194,0.063476,0.936084,-2.33,114.56,123.26
4,DE0001030708,MOT,2023-01-06,2023-01-09,RP,85.00,0,0.0,0.0,Bund Green Bond Tf 0% Ag30 Eur,...,2.99956,21.13,4017.830078,135.172394,102.066742,0.063476,0.936084,-2.33,114.56,123.26


,isincode,marketcode,referencedate,endvaluedate,pricetype,pricevalue,volume,mintoday,maxtoday,description,...,euribor_1y,vix,stoxx50,xeon,sega,interbank_stress_spread,short_yield_curve_slope,bce_fed_spread,hicp_core_lagged,hicp_headline_lagged
189505,IT0005706285,MOT,2026-05-07,NaN,LP,100.62,2074000,100.49,100.92,Btp Fx 3.8% Jul36 Eur,...,3.220984,17.08,5972.649902,149.058899,109.320000,0.027727,1.193257,-1.63,126.51,133.83
189506,IT0005707614,MOT,2026-05-04,NaN,LP,99.25,2002000,99.25,99.58,Btp Fx 3.15% Jun31 Eur,...,3.220984,18.29,5763.609863,149.038895,108.430000,0.027727,1.193257,-1.64,126.51,133.83
189507,IT0005707614,MOT,2026-05-05,NaN,LP,99.42,692000,99.27,99.44,Btp Fx 3.15% Jun31 Eur,...,3.220984,17.38,5869.629883,149.044296,108.669998,0.027727,1.193257,-1.64,126.51,133.83
189508,IT0005707614,MOT,2026-05-06,NaN,LP,100.17,2315000,99.78,100.31,Btp Fx 3.15% Jun31 Eur,...,3.220984,17.39,6027.129883,149.044098,109.290001,0.027727,1.193257,-1.64,126.51,133.83
189509,IT0005707614,MOT,2026-05-07,NaN,LP,100.21,376000,100.12,100.42,Btp Fx 3.15% Jun31 Eur,...,3.220984,17.08,5972.649902,149.058899,109.320000,0.027727,1.193257,-1.63,126.51,133.83



Statistiche:
                       referencedate     pricevalue        volume  \
count                         189510  189510.000000  1.895100e+05   
mean   2024-11-04 11:40:44.261516800      93.278877  3.628894e+06   
min              2023-01-02 00:00:00      24.160000  0.000000e+00   
25%              2024-01-24 00:00:00      86.950000  1.500000e+04   
50%              2024-11-29 00:00:00      96.060000  1.566035e+05   
75%              2025-09-04 00:00:00     101.470000  2.562000e+06   
max              2026-05-07 00:00:00     138.490000  2.025000e+09   
std                              NaN      15.306737  1.385834e+07   

            mintoday       maxtoday                 redemptiondate  \
count  189510.000000  189510.000000                         189510   
mean       78.387824      78.663350  2035-08-31 05:44:45.211334400   
min         0.000000       0.000000            2026-05-14 00:00:00   
25%        75.290000      75.720000            2028-07-04 00:00:00   
50%        93.

## 2.8 - Lagged Features per Time-Series

In [12]:
# Applica per ogni ISIN
df_ml_macro = df_ml_macro.sort_values(['isincode', 'referencedate']).reset_index(drop=True)

# Per ogni ISIN, crea lagged features per catturare la memoria storica
    # Ad es: prezzo di ieri, 3 giorni fa, 7 giorni fa, ecc.
lags = [1, 3, 7, 15, 30, 60]

for lag in lags:
    # Usando groupby().shift() pandas non usa cicli lenti, lo fa a livello C/Cython!
    df_ml_macro[f'pricevalue_lag_{lag}'] = df_ml_macro.groupby('isincode')['pricevalue'].shift(lag)

# Creiamo lag(1) anche per le macro variabili fondamentali
macro_to_lag = ['ecb_deposit_rate', 'fed_rate', 'hicp_core_lagged', 'vstoxx', 'interbank_stress_spread']
for col in macro_to_lag:
    if col in df_ml_macro.columns:
        df_ml_macro[f'{col}_lag_1'] = df_ml_macro.groupby('isincode')[col].shift(1)

# Scarta le righe con NaN nei lag (i primi 60 giorni per ogni ISIN)
lag_cols = [f'pricevalue_lag_{lag}' for lag in lags]
df_ml_macro = df_ml_macro.dropna(subset=lag_cols).copy()

print(f"✓ Lagged features creati")
print(f"Dataset dopo lagged features: {df_ml_macro.shape}")
print(f"\nColonne lag:")
lag_cols = [col for col in df_ml_macro.columns if 'lag' in col]
print(lag_cols)

✓ Lagged features creati
Dataset dopo lagged features: (171683, 40)

Colonne lag:
['hicp_core_lagged', 'hicp_headline_lagged', 'pricevalue_lag_1', 'pricevalue_lag_3', 'pricevalue_lag_7', 'pricevalue_lag_15', 'pricevalue_lag_30', 'pricevalue_lag_60', 'ecb_deposit_rate_lag_1', 'fed_rate_lag_1', 'hicp_core_lagged_lag_1', 'interbank_stress_spread_lag_1']


In [13]:
# Analogamente, crea lagged features per i dati macro
df_ml_macro['ecb_deposit_rate_lag_1'] = df_ml_macro.groupby('isincode')['ecb_deposit_rate'].shift(1)
df_ml_macro['fed_rate_lag_1'] = df_ml_macro.groupby('isincode')['fed_rate'].shift(1)
df_ml_macro['hicp_core_lagged_lag_1'] = df_ml_macro.groupby('isincode')['hicp_core_lagged'].shift(1)
df_ml_macro['hicp_headline_lagged_lag_1'] = df_ml_macro.groupby('isincode')['hicp_headline_lagged'].shift(1)

# Ancora una volta, scarta i NaN
df_ml_macro = df_ml_macro.dropna(subset=['ecb_deposit_rate_lag_1', 'fed_rate_lag_1', 'hicp_core_lagged_lag_1', 'hicp_headline_lagged_lag_1']).copy()

print(f"✓ Lagged features macro creati")
print(f"Dataset finale dopo preprocessing: {df_ml_macro.shape}")

✓ Lagged features macro creati
Dataset finale dopo preprocessing: (171393, 41)


## 2.9 - Save Preprocessed Data for Modeling

In [14]:
# Salva il dataset preprocessato e feature-engineered per il modeling
df_ml_macro.to_csv('./data/df_ml_preprocessed.csv', index=False)

print("✓ Dataset preprocessato salvato in ./data/df_ml_preprocessed.csv")
print(f"\nRiepilogo finale:")
print(f"  Numero di righe: {df_ml_macro.shape[0]}")
print(f"  Numero di colonne: {df_ml_macro.shape[1]}")
print(f"  Numero di ISIN unici: {df_ml_macro['isincode'].nunique()}")
print(f"  Data range: {df_ml_macro['referencedate'].min()} - {df_ml_macro['referencedate'].max()}")

✓ Dataset preprocessato salvato in ./data/df_ml_preprocessed.csv

Riepilogo finale:
  Numero di righe: 171393
  Numero di colonne: 41
  Numero di ISIN unici: 288
  Data range: 2023-03-30 00:00:00 - 2026-05-07 00:00:00


In [15]:
# Visualizza le feature finali disponibili per il modello
print("--- FEATURE DISPONIBILI PER IL MODELLO ---")
feature_cols = [col for col in df_ml_macro.columns if col not in ['isincode', 'description', 'referencedate', 'date', 'redemptiondate', 'endvaluedate']]
print(feature_cols)

print(f"\nTotale feature: {len(feature_cols)}")

--- FEATURE DISPONIBILI PER IL MODELLO ---
['marketcode', 'pricetype', 'pricevalue', 'volume', 'mintoday', 'maxtoday', 'coupon', 'days_to_maturity', 'years_to_maturity', 'Yield_to_Maturity', 'ecb_deposit_rate', 'ecb_mro_rate', 'fed_rate', 'esi_index', 'euribor_3m', 'euribor_1y', 'vix', 'stoxx50', 'xeon', 'sega', 'interbank_stress_spread', 'short_yield_curve_slope', 'bce_fed_spread', 'hicp_core_lagged', 'hicp_headline_lagged', 'pricevalue_lag_1', 'pricevalue_lag_3', 'pricevalue_lag_7', 'pricevalue_lag_15', 'pricevalue_lag_30', 'pricevalue_lag_60', 'ecb_deposit_rate_lag_1', 'fed_rate_lag_1', 'hicp_core_lagged_lag_1', 'interbank_stress_spread_lag_1', 'hicp_headline_lagged_lag_1']

Totale feature: 36
